In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

In [2]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('student_placement_salary_elite_v2.csv')

Saving student_placement_salary_elite_v2.csv to student_placement_salary_elite_v2.csv


In [3]:
df.info()
df.describe(include='all').T
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           9000 non-null   object 
 1   cgpa                 9000 non-null   float64
 2   branch               9000 non-null   object 
 3   college_tier         9000 non-null   int64  
 4   python_skill         9000 non-null   int64  
 5   dsa_skill            9000 non-null   int64  
 6   ml_skill             9000 non-null   int64  
 7   web_dev_skill        9000 non-null   int64  
 8   coding_score         9000 non-null   float64
 9   communication_score  9000 non-null   float64
 10  aptitude_score       9000 non-null   float64
 11  internships          9000 non-null   int64  
 12  projects             9000 non-null   int64  
 13  backlogs             9000 non-null   int64  
 14  resume_score         9000 non-null   float64
 15  skill_score          9000 non-null   i

In [4]:
df['company_type'] = df['company_type'].fillna('Not Placed')
df['job_role'] = df['job_role'].fillna('Not Placed')

In [5]:
df.info()
df.describe(include='all').T
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           9000 non-null   object 
 1   cgpa                 9000 non-null   float64
 2   branch               9000 non-null   object 
 3   college_tier         9000 non-null   int64  
 4   python_skill         9000 non-null   int64  
 5   dsa_skill            9000 non-null   int64  
 6   ml_skill             9000 non-null   int64  
 7   web_dev_skill        9000 non-null   int64  
 8   coding_score         9000 non-null   float64
 9   communication_score  9000 non-null   float64
 10  aptitude_score       9000 non-null   float64
 11  internships          9000 non-null   int64  
 12  projects             9000 non-null   int64  
 13  backlogs             9000 non-null   int64  
 14  resume_score         9000 non-null   float64
 15  skill_score          9000 non-null   i

# Features

In [6]:
# Experience Score (internships weighted + skill depth)
df['experience_score'] = df['internships'] * 2 + df['skill_score']

# Academic Score (CGPA scaled, penalized by backlogs)
df['academic_score'] = df['cgpa'] - (df['backlogs'] * 0.5)

# Employability Index (composite)
df['employability_index'] = (
    df['academic_score'] * 0.4 +
    df['skill_score'] * 0.3 +
    df['internships'] * 0.2 +
    (4 - df['college_tier']) * 0.1
)

# Total technical score (skills + internship weight)
df['total_technical_score'] = df['skill_score'] + df['internships'] * 1.5

# Academic-experience index (CGPA blended with experience, penalized by backlogs)
df['academic_experience_index'] = (
    df['cgpa'] * 0.5 +
    df['internships'] * 0.3 +
    df['skill_score'] * 0.2 -
    df['backlogs'] * 0.4
)

# Tier-adjusted CGPA (Tier 1 students get a small boost effect)
df['tier_adjusted_cgpa'] = df['cgpa'] * (4 - df['college_tier']) / 3

# Interaction: skills × internships
df['skill_internship_interaction'] = df['skill_score'] * df['internships']

# Backlog-free flag
df['no_backlogs'] = (df['backlogs'] == 0).astype(int)

df[['experience_score','academic_score','employability_index','total_technical_score','academic_experience_index',
    'tier_adjusted_cgpa','skill_internship_interaction','no_backlogs']].describe()

,experience_score,academic_score,employability_index,total_technical_score,academic_experience_index,tier_adjusted_cgpa,skill_internship_interaction,no_backlogs
count,9000.000000,9000.000000,9000.000000,9000.000000,9000.000000,9000.000000,9000.000000,9000.000000
mean,3.962889,7.296547,3.893508,3.446889,4.274451,4.975294,1.947222,0.705444
std,2.143056,1.499026,0.693081,1.723791,0.855737,2.298055,2.268212,0.455868
min,0.000000,3.510000,1.936000,0.000000,1.540000,1.670000,0.000000,0.000000
25%,2.000000,6.040000,3.376000,2.000000,3.635000,2.893333,0.000000,0.000000
50%,4.000000,7.310000,3.884000,3.500000,4.275000,4.920000,2.000000,1.000000
75%,5.000000,8.560000,4.412000,4.500000,4.940000,6.506667,3.000000,1.000000
max,10.000000,10.000000,5.912000,8.500000,6.680000,10.000000,12.000000,1.000000


# Categorical Encoding

In [7]:
# Drop identifier
df_enc = df.drop(columns=['student_id'])

# One-hot encode branch and company_type
df_enc['company_type'] = df_enc['company_type'].fillna('None')

df_enc = pd.get_dummies(df_enc, columns=['branch','company_type'],
                        drop_first=True, dtype=int)

if 'job_role' in df_enc.columns:
    df_enc = df_enc.drop(columns=['job_role'])

print(df_enc.shape)
df_enc.head()

(9000, 33)


,cgpa,college_tier,python_skill,dsa_skill,ml_skill,web_dev_skill,coding_score,communication_score,aptitude_score,internships,projects,backlogs,resume_score,skill_score,placed,salary_lpa,experience_score,academic_score,employability_index,total_technical_score,academic_experience_index,tier_adjusted_cgpa,skill_internship_interaction,no_backlogs,branch_Civil,branch_ECE,branch_EEE,branch_IT,branch_Mechanical,company_type_Mid-size,company_type_Not Placed,company_type_Startup,company_type_Top Tech
0,6.87,1,1,1,0,0,15.6,4.3,92.0,1,3,0,62.6,2,1,63.55,4,6.87,3.848,3.5,4.135,6.870000,2,1,1,0,0,0,0,0,0,0,0
1,6.52,2,1,0,0,1,13.9,5.8,62.0,1,6,0,77.5,2,1,75.17,4,6.52,3.608,3.5,3.960,4.346667,2,1,1,0,0,0,0,0,0,0,0
2,5.33,1,1,1,1,0,9.8,8.1,66.4,0,5,1,76.0,3,1,80.44,3,4.83,3.132,3.0,2.865,5.330000,0,0,0,0,0,1,0,0,0,0,0
3,6.04,3,1,0,1,0,39.5,9.6,83.6,0,6,0,74.3,2,1,72.11,2,6.04,3.116,2.0,3.420,2.013333,0,1,1,0,0,0,0,0,0,0,0
4,6.78,2,0,1,0,1,7.5,9.9,86.3,0,3,0,66.8,2,1,67.05,2,6.78,3.512,2.0,3.790,4.520000,0,1,0,0,0,0,1,1,0,0,0


# Datasets

In [9]:
# Classification Dataset
leakage_cols = [c for c in df_enc.columns if c.startswith('company_type_')] + ['salary_lpa']
X_clf = df_enc.drop(columns=['placed'] + leakage_cols)
y_clf = df_enc['placed']

print("Classification X:", X_clf.shape, "| y:", y_clf.shape)

Classification X: (9000, 27) | y: (9000,)


In [10]:
# Regression Dataset
placed_df = df_enc[df_enc['placed'] == 1].copy()

X_reg = placed_df.drop(columns=['placed','salary_lpa'])
y_reg = placed_df['salary_lpa']

print("Regression X:", X_reg.shape, "| y:", y_reg.shape)

Regression X: (7702, 31) | y: (7702,)


# Splitting Dataset

In [11]:
# --- Classification ---
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

scaler_clf = StandardScaler()
X_clf_train_scaled = scaler_clf.fit_transform(X_clf_train)
X_clf_test_scaled  = scaler_clf.transform(X_clf_test)

# --- Regression ---
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

scaler_reg = StandardScaler()
X_reg_train_scaled = scaler_reg.fit_transform(X_reg_train)
X_reg_test_scaled  = scaler_reg.transform(X_reg_test)

print("CLF train/test:", X_clf_train.shape, X_clf_test.shape)
print("REG train/test:", X_reg_train.shape, X_reg_test.shape)

CLF train/test: (7200, 27) (1800, 27)
REG train/test: (6161, 31) (1541, 31)
